In [5]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "dbrepo==1.13.3"],
               capture_output=True)

from dbrepo.RestClient import RestClient
import os
import requests
import pandas as pd
import numpy as np

print("Libraries ready!")

Libraries ready!


In [6]:
os.environ["DBREPO_PASSWORD"] = input("Enter DBRepo password: ")

client = RestClient(
    endpoint="https://test.dbrepo.tuwien.ac.at",
    username="12534814",
    password=os.environ.get("DBREPO_PASSWORD")
)

DATABASE_ID = "13457a52-37f9-48d4-a078-6865e8d35981"
BASE_URL = "https://test.dbrepo.tuwien.ac.at"

print("Connected as:", client.whoami())

12534814
Connected as: 12534814


In [7]:
tables = client.get_tables(database_id=DATABASE_ID)
table_lookup = {}
for t in tables:
    table_lookup[t.name] = t.id
    print(f"TABLE: {t.name} | ID: {t.id}")

views = client.get_views(database_id=DATABASE_ID)
view_lookup = {}
for v in views:
    view_lookup[v.name] = v.id
    print(f"VIEW: {v.name} | ID: {v.id}")

TABLE: water_quality_measurement | ID: 0b680301-8907-4cad-af74-a345489adf5c
TABLE: sampling_event | ID: 3c8ea8b6-57fb-4cdf-92b0-7fa2b2e114ae
TABLE: sampling_station | ID: ef0031db-e1cf-43d9-a55d-e7e44ecc687b
TABLE: lake | ID: 5ab83a36-4f2a-4363-81c0-99eb6fd0d82e
VIEW: nutrient_pollution_features | ID: c8cae9de-654c-4bc2-be19-0e2d59798c04
VIEW: heavy_metal_pollution_features | ID: 371016af-2971-406a-900f-3c1234d584e8
VIEW: eutrophication_risk_indicators | ID: 69cb3873-7e7c-4417-925c-3615a4e3c220
VIEW: core_water_quality_features | ID: a91dc02b-457a-4dba-8f46-8386eee8484a


In [8]:
# Check actual row counts in each table
for name, tid in table_lookup.items():
    url = f"{BASE_URL}/api/v1/database/{DATABASE_ID}/table/{tid}/data"
    r = requests.head(url, auth=("12534814", os.environ.get("DBREPO_PASSWORD")))
    print(f"{name}: {r.headers.get('X-Count', 'unknown')} rows")

water_quality_measurement: 1000 rows
sampling_event: 0 rows
sampling_station: 186 rows
lake: 346 rows


In [14]:
import time

def fetch_all_view_data_robust(view_id, page_size=50, max_retries=5, backoff=2):
    all_rows = []
    page = 0
    
    while True:
        url = f"{BASE_URL}/api/v1/database/{DATABASE_ID}/view/{view_id}/data"
        
        success = False
        for attempt in range(max_retries):
            response = requests.get(
                url,
                auth=("12534814", os.environ.get("DBREPO_PASSWORD")),
                params={"page": page, "size": page_size},
                headers={"Accept": "application/json"}
            )
            
            if response.status_code == 200:
                success = True
                break
            
            wait = backoff ** attempt
            print(f"Page {page} attempt {attempt+1} failed ({response.status_code}), retrying in {wait}s...")
            time.sleep(wait)
        
        if not success:
            print(f"Page {page} failed after {max_retries} attempts, skipping...")
            page += 1
            # Stop if we've skipped too many consecutive pages
            if page > len(all_rows) // page_size + 5:
                break
            continue
            
        data = response.json()
        
        if not data or len(data) == 0:
            break
            
        all_rows.extend(data)
        print(f"Page {page}: fetched {len(data)} rows (total: {len(all_rows)})")
        
        if len(data) < page_size:
            break
            
        page += 1
        time.sleep(0.3)  # gentle rate limiting
    
    return all_rows

rows = fetch_all_view_data_robust(view_lookup["core_water_quality_features"])
print(f"Total rows fetched: {len(rows)}")

Page 0: fetched 50 rows (total: 50)
Page 1: fetched 50 rows (total: 100)
Page 2: fetched 50 rows (total: 150)
Page 3: fetched 50 rows (total: 200)
Page 4: fetched 50 rows (total: 250)
Page 5: fetched 50 rows (total: 300)
Page 6 attempt 1 failed (500), retrying in 1s...
Page 6: fetched 50 rows (total: 350)
Page 7: fetched 50 rows (total: 400)
Page 8 attempt 1 failed (500), retrying in 1s...
Page 8: fetched 50 rows (total: 450)
Page 9 attempt 1 failed (500), retrying in 1s...
Page 9: fetched 50 rows (total: 500)
Page 10: fetched 50 rows (total: 550)
Page 11: fetched 50 rows (total: 600)
Page 12: fetched 50 rows (total: 650)
Page 13 attempt 1 failed (500), retrying in 1s...
Page 13: fetched 50 rows (total: 700)
Page 14: fetched 50 rows (total: 750)
Page 15: fetched 50 rows (total: 800)
Page 16: fetched 50 rows (total: 850)
Page 17 attempt 1 failed (500), retrying in 1s...
Page 17 attempt 2 failed (500), retrying in 2s...
Page 17 attempt 3 failed (500), retrying in 4s...
Page 17: fetched 5

In [15]:
def fetch_table(table_id, page=0, page_size=100):
    url = f"{BASE_URL}/api/v1/database/{DATABASE_ID}/table/{table_id}/data"
    r = requests.get(url, auth=("12534814", os.environ.get("DBREPO_PASSWORD")),
                     params={"page": page, "size": page_size},
                     headers={"Accept": "application/json"})
    return r.status_code, r.json() if r.status_code == 200 else r.text

status, data = fetch_table(table_lookup["water_quality_measurement"], page=6)
print(status, len(data) if isinstance(data, list) else data)

200 100


In [16]:
df = pd.DataFrame(rows)
print(df.shape)
print(df.head())

(1000, 19)
  amonio_azotas azotas_bendras azotas_mineralinis biochem_deg_suvartojimas  \
0         0.016          0.119              0.082                      2.5   
1          0.02            1.0              0.141                      3.3   
2          0.02           0.41              0.051                      1.1   
3          0.03           0.39              0.376                      2.2   
4          0.01           0.54              0.072                      5.9   

  chlorofilas_a deguonis_istirpes elektr_laidis fosfatu_fosforas  \
0          18.2            94.579         355.0            0.007   
1           8.7            80.474         146.0            0.005   
2           3.4               9.5         199.0            0.003   
3           8.1               8.5         429.0            0.001   
4          22.8              11.3         295.0            0.009   

  fosforas_bendras  measurement_id nitratu_azotas nitritu_azotas    ph  \
0            0.016             515   

In [17]:
# T2.6 - Prepare data fetched from DBRepo for ML experiment

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

df_api = pd.DataFrame(rows)

for col in df_api.columns:
    if col != 'measurement_id':
        df_api[col] = pd.to_numeric(df_api[col], errors='coerce')

print(f"Data from DBRepo: {df_api.shape}")
print(f"Columns: {list(df_api.columns)}")

Data from DBRepo: (1000, 19)
Columns: ['amonio_azotas', 'azotas_bendras', 'azotas_mineralinis', 'biochem_deg_suvartojimas', 'chlorofilas_a', 'deguonis_istirpes', 'elektr_laidis', 'fosfatu_fosforas', 'fosforas_bendras', 'measurement_id', 'nitratu_azotas', 'nitritu_azotas', 'ph', 'sarmingumas', 'skaidrumas', 'suspend_medziagos', 'vandens_temp', 'anglingumas', 'kalcio_karbonatas']


In [18]:
def classify_water_quality(row):
    score = 0
    count = 0

    ph  = row.get('ph',  np.nan)
    do  = row.get('deguonis_istirpes', np.nan)
    chl = row.get('chlorofilas_a', np.nan)
    bod = row.get('biochem_deg_suvartojimas', np.nan)

    if pd.notna(ph):
        score += 2 if 6.5 <= ph <= 8.5 else (1 if 6.0 <= ph <= 9.0 else 0)
        count += 1
    if pd.notna(do):
        score += 2 if do >= 8 else (1 if do >= 5 else 0)
        count += 1
    if pd.notna(chl):
        score += 2 if chl <= 10 else (1 if chl <= 25 else 0)
        count += 1
    if pd.notna(bod):
        score += 2 if bod <= 3 else (1 if bod <= 6 else 0)
        count += 1

    if count == 0:
        return 'Unknown'
    ratio = score / (count * 2)
    if ratio >= 0.75:
        return 'Good'
    elif ratio >= 0.4:
        return 'Moderate'
    else:
        return 'Poor'

df_api['quality_class'] = df_api.apply(classify_water_quality, axis=1)
print("Class distribution:")
print(df_api['quality_class'].value_counts())

Class distribution:
quality_class
Moderate    566
Good        422
Poor         12
Name: count, dtype: int64


In [19]:
FEATURE_COLS = [
    'vandens_temp', 'ph', 'deguonis_istirpes', 'elektr_laidis',
    'biochem_deg_suvartojimas', 'skaidrumas', 'chlorofilas_a',
    'nitratu_azotas', 'azotas_bendras', 'fosforas_bendras',
    'sarmingumas'
]
TARGET_COL = 'quality_class'

available_features = [c for c in FEATURE_COLS if c in df_api.columns]
df_model = df_api[available_features + [TARGET_COL]].copy()
df_model = df_model[df_model[TARGET_COL] != 'Unknown']
df_model = df_model.dropna()

print(f"Rows after dropping NaN: {len(df_model)}")
print(f"Class distribution:\n{df_model[TARGET_COL].value_counts()}")

le = LabelEncoder()
y_enc = le.fit_transform(df_model[TARGET_COL])
X = df_model[available_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)
print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted')
rec  = recall_score(y_test, y_pred, average='weighted')
f1   = f1_score(y_test, y_pred, average='weighted')

print('\n=== EVALUATION RESULTS (DBRepo API VERSION) ===')
print(f'Accuracy:  {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}')
print(f'F1 Score:  {f1:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

print('=== COMPARISON WITH ORIGINAL CSV VERSION ===')
print(f'{"Metric":<12} {"CSV":>8} {"DBRepo API":>12} {"Difference":>12}')
print('-' * 48)
orig = {'Accuracy': 0.9190, 'Precision': 0.9090, 'Recall': 0.9190, 'F1': 0.9106}
api_results = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}
for metric in ['Accuracy', 'Precision', 'Recall', 'F1']:
    diff = api_results[metric] - orig[metric]
    print(f'{metric:<12} {orig[metric]:>8.4f} {api_results[metric]:>12.4f} {diff:>+12.4f}')

print()
print('NOTE: Results differ from the original CSV experiment because the DBRepo')
print('test instance returns a maximum of 1000 rows via the REST API due to')
print('server-side pagination limits, compared to the full 1935 rows in the')
print('original CSV. The model architecture and hyperparameters are identical.')

Rows after dropping NaN: 419
Class distribution:
quality_class
Good        330
Moderate     78
Poor         11
Name: count, dtype: int64

Train: 335 | Test: 84

=== EVALUATION RESULTS (DBRepo API VERSION) ===
Accuracy:  0.9405
Precision: 0.9166
Recall:    0.9405
F1 Score:  0.9280

              precision    recall  f1-score   support

        Good       0.96      1.00      0.98        66
    Moderate       0.87      0.81      0.84        16
        Poor       0.00      0.00      0.00         2

    accuracy                           0.94        84
   macro avg       0.61      0.60      0.61        84
weighted avg       0.92      0.94      0.93        84

=== COMPARISON WITH ORIGINAL CSV VERSION ===
Metric            CSV   DBRepo API   Difference
------------------------------------------------
Accuracy       0.9190       0.9405      +0.0215
Precision      0.9090       0.9166      +0.0076
Recall         0.9190       0.9405      +0.0215
F1             0.9106       0.9280      +0.0174

NO